# Assignment 4: Types of Supervised Machine Learning

In this assignment, I will train and compare three different models for each type of supervised learning:
- Classification (Wine dataset): Logistic Regression, Support Vector Classifier (SVC), and K-Nearest Neighbors Classifier
- Regression (Car Prices dataset): Linear Regression, Support Vector Regressor (SVR), and K-Nearest Neighbors Regressor

For each dataset, I will load and explore the data, split it into training and test sets, train the three models, and compare their performance. Then I will tune the hyperparameters of each model and report how the changes affect performance.

Note: The Wine dataset classes are three different grape cultivars, not quality levels. For the regression models, I use R² and RMSE (Root Mean Squared Error) instead of accuracy, because accuracy only applies to classification.

# Part A: Classification with the Wine Dataset

## Step 1: Load and Explore the Dataset

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Classification models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# Regression models (for Part B)
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
pd.set_option("display.max_columns", None)


In [3]:
# Load the Wine dataset
wine = load_wine(as_frame=True)
wine_df = wine.frame

In [4]:
print("Shape:", wine_df.shape)

Shape: (178, 14)


In [7]:
print("\nClass counts:")
print(wine_df["target"].value_counts().sort_index())



Class counts:
target
0    59
1    71
2    48
Name: count, dtype: int64


In [8]:
print("\nMissing values:", wine_df.isnull().sum().sum())
wine_df.describe().round(2)


Missing values: 0


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
count,178.00,178.00,178.00,178.00,178.00,178.00,178.00,178.00,178.00,178.00,178.00,178.00,178.00,178.00
mean,13.00,2.34,2.37,19.49,99.74,2.30,2.03,0.36,1.59,5.06,0.96,2.61,746.89,0.94
std,0.81,1.12,0.27,3.34,14.28,0.63,1.00,0.12,0.57,2.32,0.23,0.71,314.91,0.78
min,11.03,0.74,1.36,10.60,70.00,0.98,0.34,0.13,0.41,1.28,0.48,1.27,278.00,0.00
25%,12.36,1.60,2.21,17.20,88.00,1.74,1.20,0.27,1.25,3.22,0.78,1.94,500.50,0.00
50%,13.05,1.87,2.36,19.50,98.00,2.36,2.13,0.34,1.56,4.69,0.96,2.78,673.50,1.00
75%,13.68,3.08,2.56,21.50,107.00,2.80,2.88,0.44,1.95,6.20,1.12,3.17,985.00,2.00
max,14.83,5.80,3.23,30.00,162.00,3.88,5.08,0.66,3.58,13.00,1.71,4.00,1680.00,2.00


### Interpretation

I imported all the libraries for both parts of the assignment, including the three classification models (Logistic Regression, SVC, and KNN Classifier), the three regression models (Linear Regression, SVR, and KNN Regressor), GridSearchCV for hyperparameter tuning, and the evaluation metrics.

The Wine dataset has 178 rows and 14 columns: 13 chemical features and 1 target column. There are no missing values.

The three classes are:
- Class 0: 59 wines
- Class 1: 71 wines
- Class 2: 48 wines

The classes are fairly balanced, so I do not need techniques like SMOTE. From the summary statistics, the features are on very different scales (for example, proline is in the hundreds or thousands, while hue is below 2). This matters for SVC and KNN, which are sensitive to feature scale, so I will scale the features before training.

## Step 2: Split Data into Training and Test Sets

In [9]:
X_wine = wine_df.drop("target", axis=1)
y_wine = wine_df["target"]

X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42, stratify=y_wine
)

print("Training set:", X_train_w.shape)
print("Test set:", X_test_w.shape)

Training set: (142, 13)
Test set: (36, 13)


## Steps 3 to 5: Train Multiple Models and Evaluate Their Performance

In [11]:
# Each model is a Pipeline: scale first, then the model
clf_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "SVC": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(random_state=42))
    ]),
    "KNN Classifier": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier())
    ])
}

clf_results = []

for name, model in clf_models.items():
    model.fit(X_train_w, y_train_w)                 
    y_pred = model.predict(X_test_w)              
    train_acc = accuracy_score(y_train_w, model.predict(X_train_w))
    test_acc = accuracy_score(y_test_w, y_pred)
    clf_results.append({"Model": name,
                        "Train Accuracy": round(train_acc, 4),
                        "Test Accuracy": round(test_acc, 4)})
    print(f"===== {name} =====")
    print(classification_report(y_test_w, y_pred, target_names=wine.target_names))

clf_results_df = pd.DataFrame(clf_results)
print(clf_results_df)

===== Logistic Regression =====
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        12
     class_1       0.93      1.00      0.97        14
     class_2       1.00      0.90      0.95        10

    accuracy                           0.97        36
   macro avg       0.98      0.97      0.97        36
weighted avg       0.97      0.97      0.97        36

===== SVC =====
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        12
     class_1       0.93      1.00      0.97        14
     class_2       1.00      0.90      0.95        10

    accuracy                           0.97        36
   macro avg       0.98      0.97      0.97        36
weighted avg       0.97      0.97      0.97        36

===== KNN Classifier =====
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        12
     class_1       1.00      0.93      0.96        14


### Interpretation

I trained three classification models using their default settings. Each model was placed in a Pipeline with StandardScaler, so the features were scaled using only the training data.

Results:
- Logistic Regression: training accuracy 100%, test accuracy 97.22%
- SVC: training accuracy 99.30%, test accuracy 97.22%
- KNN Classifier (k = 5): training accuracy 97.89%, test accuracy 97.22%

All three models reached the same test accuracy of 97.22%, which means each one classified 35 out of 36 test wines correctly and made only one mistake. The mistakes were different:
- Logistic Regression and SVC both misclassified one class_2 wine as class_1. This is why their class_2 recall is 0.90 and class_1 precision is 0.93.
- KNN misclassified one class_1 wine as class_2. This is why its class_1 recall is 0.93 and class_2 precision is 0.91.

Which model performs best?
On the test set, the three models are tied, so I cannot pick a clear winner from these results alone. The test set has only 36 wines, so a single prediction changes accuracy by about 2.8%.

Looking at training accuracy, Logistic Regression scored 100% on the training data, which could be a small sign of overfitting, while KNN had the smallest gap between training and test accuracy. All three models perform well because the Wine dataset has clear chemical differences between the three cultivars, and scaling the features helped SVC and KNN in particular, since they depend on distances.

To compare the models more fairly, I will use cross-validation and hyperparameter tuning in the next step.

## Step 6: Parameter Tuning and Optimization

In [13]:
# Hyperparameters to try for each model
clf_param_grids = {
    "Logistic Regression": {
        "model__C": [0.01, 0.1, 1, 10, 100]
    },
    "SVC": {
        "model__C": [0.1, 1, 10, 100],
        "model__kernel": ["linear", "rbf"]
    },
    "KNN Classifier": {
        "model__n_neighbors": [1, 3, 5, 7, 9, 11, 15, 21],
        "model__weights": ["uniform", "distance"]
    }
}

clf_tuned_results = []

for name, model in clf_models.items():
    grid = GridSearchCV(model, clf_param_grids[name], cv=5, scoring="accuracy")
    grid.fit(X_train_w, y_train_w)

    test_acc = accuracy_score(y_test_w, grid.predict(X_test_w))
    clf_tuned_results.append({"Model": name,
                              "Best Parameters": grid.best_params_,
                              "Best CV Accuracy": round(grid.best_score_, 4),
                              "Test Accuracy": round(test_acc, 4)})

    # Show how each parameter setting performed
    print(f"===== {name} =====")
    cv_table = pd.DataFrame(grid.cv_results_)[["params", "mean_test_score"]]
    cv_table["mean_test_score"] = cv_table["mean_test_score"].round(4)
    print(cv_table.to_string(index=False))
    print()

clf_tuned_df = pd.DataFrame(clf_tuned_results)
print(clf_tuned_df.to_string(index=False))

===== Logistic Regression =====
            params  mean_test_score
{'model__C': 0.01}           0.9791
 {'model__C': 0.1}           0.9931
   {'model__C': 1}           0.9931
  {'model__C': 10}           0.9860
 {'model__C': 100}           0.9860

===== SVC =====
                                      params  mean_test_score
{'model__C': 0.1, 'model__kernel': 'linear'}           0.9862
   {'model__C': 0.1, 'model__kernel': 'rbf'}           0.9793
  {'model__C': 1, 'model__kernel': 'linear'}           0.9862
     {'model__C': 1, 'model__kernel': 'rbf'}           0.9862
 {'model__C': 10, 'model__kernel': 'linear'}           0.9862
    {'model__C': 10, 'model__kernel': 'rbf'}           0.9862
{'model__C': 100, 'model__kernel': 'linear'}           0.9862
   {'model__C': 100, 'model__kernel': 'rbf'}           0.9862

===== KNN Classifier =====
                                                  params  mean_test_score
  {'model__n_neighbors': 1, 'model__weights': 'uniform'}           0.9367
 

### Interpretation

I used GridSearchCV with 5-fold cross-validation on the training set to test different hyperparameters for each model. GridSearchCV tries every combination, picks the one with the highest average cross-validation accuracy, and then retrains that best version on the full training set. I only used the test set at the end to check the final model.

Logistic Regression (C = regularization strength, where a smaller C means stronger regularization):
- C = 0.01 gave the lowest accuracy (97.91%). The regularization was too strong, so the model became too simple (underfitting).
- C = 0.1 and C = 1 gave the highest accuracy (99.31%).
- C = 10 and C = 100 dropped slightly to 98.60%. With weak regularization, the model fits the training data too closely (slight overfitting).
- The best setting was C = 0.1, which reached 100% accuracy on the test set.

SVC (C and kernel):
- Almost every combination gave the same accuracy of 98.62%. The only lower one was C = 0.1 with the rbf kernel (97.93%).
- This shows SVC was not very sensitive to these parameters on this dataset.
- The linear kernel performed just as well as the rbf (curved) kernel, which suggests the three wine classes can be separated with straight boundaries.
- The best setting chosen was C = 0.1 with a linear kernel. Its test accuracy stayed at 97.22%, the same as before tuning.

KNN Classifier (k and weights):
- k = 1 gave the lowest accuracy (93.67%), because one neighbour makes the prediction depend on a single wine (overfitting).
- Accuracy improved as k increased, reaching 97.22% at k = 11 and k = 15.
- The weights setting (uniform vs distance) made almost no difference.
- The best setting was k = 11 with uniform weights, which reached 100% accuracy on the test set.

Which model performs best and why?
Logistic Regression with C = 0.1 performed best overall. It had the highest cross-validation accuracy (99.31%) and 100% test accuracy. Because cross-validation uses all the training data in 5 rounds, it is a more reliable comparison than the small 36-wine test set. Logistic Regression works well here because the wine classes are close to linearly separable (the linear SVC kernel also did well), and the right amount of regularization stops it from overfitting. KNN also reached 100% on the test set, but its cross-validation accuracy (97.22%) was lower, so it is slightly less reliable.

# Part B: Regression with the Car Prices Dataset

## Steps 1 and 2: Load, Explore, and Split the Dataset

In [14]:
# Load the data
car_df = pd.read_csv("CarPrice_Assignment.csv")


In [15]:
print("Original shape:", car_df.shape)
print("Missing values:", car_df.isnull().sum().sum())

Original shape: (205, 26)
Missing values: 0


In [16]:
print("\nPrice statistics:")
print(car_df["price"].describe().round(2))


Price statistics:
count      205.00
mean     13276.71
std       7988.85
min       5118.00
25%       7788.00
50%      10295.00
75%      16503.00
max      45400.00
Name: price, dtype: float64


In [17]:
# Drop unhelpful columns and one-hot encode text columns
car_clean = car_df.drop(["car_ID", "CarName"], axis=1)
car_encoded = pd.get_dummies(car_clean, drop_first=True, dtype=int)
print("\nShape after encoding:", car_encoded.shape)


Shape after encoding: (205, 44)


In [18]:
# Split into features and target, then 80/20
X_car = car_encoded.drop("price", axis=1)
y_car = car_encoded["price"]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_car, y_car, test_size=0.2, random_state=42
)
print("\nTraining set:", X_train_c.shape)
print("Test set:", X_test_c.shape)


Training set: (164, 43)
Test set: (41, 43)


### Interpretation

I loaded the Car Prices dataset, which has 205 cars and 26 columns with no missing values. Car prices range from $5,118 to $45,400, with a median of $10,295 and a mean of $13,276.71. The mean is higher than the median, so prices are right-skewed (a few expensive cars pull the average up).

I prepared the data the same way as in Assignment 3:
- I removed car_ID (just a row number) and CarName (147 unique values out of 205 cars).
- I used one-hot encoding with drop_first=True to convert the 9 text columns into 0/1 columns. This increased the data from 24 to 44 columns.

I then split the data into 43 features (X) and the price (y), and used an 80/20 split with random_state=42. The training set has 164 cars and the test set has 41 cars. I did not use stratify, because price is a continuous number.

Note: the assignment mentions features like year and mileage, but this dataset does not include them. It uses car attributes such as engine size, horsepower, car dimensions, weight, fuel type, and body type.

## Steps 3 to 5: Train Multiple Models and Evaluate Their Performance

In [19]:
reg_models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "SVR": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR())
    ]),
    "KNN Regressor": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor())
    ])
}

reg_results = []

for name, model in reg_models.items():
    model.fit(X_train_c, y_train_c)
    y_pred = model.predict(X_test_c)
    reg_results.append({
        "Model": name,
        "Train R²": round(r2_score(y_train_c, model.predict(X_train_c)), 4),
        "Test R²": round(r2_score(y_test_c, y_pred), 4),
        "Test RMSE ($)": round(np.sqrt(mean_squared_error(y_test_c, y_pred)), 2)
    })

reg_results_df = pd.DataFrame(reg_results)
print(reg_results_df.to_string(index=False))

            Model  Train R²  Test R²  Test RMSE ($)
Linear Regression    0.9476   0.8926        2912.39
              SVR   -0.1082  -0.0999        9318.25
    KNN Regressor    0.8249   0.7879        4091.49


### Interpretation

I trained three regression models with their default settings, each in a Pipeline with StandardScaler. For regression, I used R² and RMSE instead of accuracy. R² shows how much of the variation in price the model explains (1 is perfect), and RMSE shows the average prediction error in dollars.

Results:
- Linear Regression: training R² 0.9476, test R² 0.8926, test RMSE $2,912.39
- SVR: training R² -0.1082, test R² -0.0999, test RMSE $9,318.25
- KNN Regressor (k = 5): training R² 0.8249, test R² 0.7879, test RMSE $4,091.49

Which model performs best and why?
Linear Regression performed best by far, explaining about 89% of the variation in car prices with an average error of about $2,912. The results are the same as in Assignment 3, because I used the same split. Linear Regression works well here because many important features, such as engine size, car width, and curb weight, have a fairly straight-line relationship with price. Its training R² is a little higher than its test R², which shows mild overfitting.

KNN Regressor came second, with a test R² of 0.7879 and an average error of about $4,091. KNN predicts a car's price by averaging the prices of its 5 most similar cars. With only 164 training cars and 43 features, many cars do not have very similar neighbours, especially expensive ones, so the averages are less accurate.

SVR performed very badly, with a negative R² on both the training and test sets. A negative R² means the model is worse than simply predicting the average price for every car. This happened because of the default setting C = 1. C controls how much SVR is penalized for errors. Car prices are in the thousands of dollars, so C = 1 is far too small, and SVR barely tries to fit the data. This does not mean SVR is a bad algorithm. It shows that SVR needs its hyperparameters tuned, which I will do in the next step.

## Step 6: Parameter Tuning and Optimization

Plain Linear Regression does not have hyperparameters to tune, so I used Ridge Regression, which is Linear Regression with L2 regularization. Its alpha parameter controls the regularization strength.

In [20]:
from sklearn.linear_model import Ridge

In [21]:
reg_tune_models = {
    "Ridge (Linear Regression)": Pipeline([("scaler", StandardScaler()), ("model", Ridge())]),
    "SVR": Pipeline([("scaler", StandardScaler()), ("model", SVR())]),
    "KNN Regressor": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsRegressor())])
}

reg_param_grids = {
    "Ridge (Linear Regression)": {
        "model__alpha": [0.01, 0.1, 1, 10, 100]
    },
    "SVR": {
        "model__C": [1, 100, 1000, 10000, 100000],
        "model__kernel": ["linear", "rbf"]
    },
    "KNN Regressor": {
        "model__n_neighbors": [1, 2, 3, 5, 7, 9],
        "model__weights": ["uniform", "distance"]
    }
}

reg_tuned_results = []

for name, model in reg_tune_models.items():
    grid = GridSearchCV(model, reg_param_grids[name], cv=5, scoring="r2")
    grid.fit(X_train_c, y_train_c)

    y_pred = grid.predict(X_test_c)
    reg_tuned_results.append({
        "Model": name,
        "Best Parameters": grid.best_params_,
        "Best CV R²": round(grid.best_score_, 4),
        "Test R²": round(r2_score(y_test_c, y_pred), 4),
        "Test RMSE ($)": round(np.sqrt(mean_squared_error(y_test_c, y_pred)), 2)
    })

    print(f"===== {name} =====")
    cv_table = pd.DataFrame(grid.cv_results_)[["params", "mean_test_score"]]
    cv_table["mean_test_score"] = cv_table["mean_test_score"].round(4)
    print(cv_table.to_string(index=False))
    print()

reg_tuned_df = pd.DataFrame(reg_tuned_results)
print(reg_tuned_df.to_string(index=False))

===== Ridge (Linear Regression) =====
                params  mean_test_score
{'model__alpha': 0.01}           0.8338
 {'model__alpha': 0.1}           0.8409
   {'model__alpha': 1}           0.8678
  {'model__alpha': 10}           0.8796
 {'model__alpha': 100}           0.8232

===== SVR =====
                                         params  mean_test_score
     {'model__C': 1, 'model__kernel': 'linear'}           0.1003
        {'model__C': 1, 'model__kernel': 'rbf'}          -0.1356
   {'model__C': 100, 'model__kernel': 'linear'}           0.8037
      {'model__C': 100, 'model__kernel': 'rbf'}           0.0579
  {'model__C': 1000, 'model__kernel': 'linear'}           0.8445
     {'model__C': 1000, 'model__kernel': 'rbf'}           0.4386
 {'model__C': 10000, 'model__kernel': 'linear'}           0.8080
    {'model__C': 10000, 'model__kernel': 'rbf'}           0.7623
{'model__C': 100000, 'model__kernel': 'linear'}           0.7598
   {'model__C': 100000, 'model__kernel': 'rbf'}        

### Interpretation

I used GridSearchCV with 5-fold cross-validation on the training set, using R² as the score, to tune each regression model.

Ridge Regression (alpha = regularization strength, where a larger alpha means stronger regularization):
- Small alpha values (0.01 and 0.1) gave lower CV R² (0.8338 and 0.8409). With very little regularization, Ridge behaves almost like plain Linear Regression and overfits.
- CV R² improved as alpha increased, reaching the best value of 0.8796 at alpha = 10.
- At alpha = 100, CV R² dropped to 0.8232, because the regularization was too strong and the model became too simple (underfitting).
- Ridge with alpha = 10 had a test R² of 0.8555, which is lower than plain Linear Regression's 0.8926. However, Ridge with alpha = 0.01 (very close to plain Linear Regression) only scored 0.8338 in cross-validation. This suggests regularization helps the model generalize better overall, and plain Linear Regression was lucky on this particular 41-car test set.

SVR (C and kernel):
- With C = 1, SVR performed very poorly (CV R² 0.1003 for linear and -0.1356 for rbf), confirming that the default C was far too small for prices in the thousands.
- Increasing C greatly improved performance. With the linear kernel, CV R² rose to 0.8037 at C = 100 and peaked at 0.8445 at C = 1000.
- Beyond that, the linear kernel got worse (0.7598 at C = 100,000), because a very large C makes the model try too hard to fit every training car (overfitting).
- The rbf kernel needed much larger C values to perform well, reaching 0.8071 at C = 100,000.
- The best setting was C = 1000 with the linear kernel. Test R² improved from -0.0999 to 0.8808, and RMSE dropped from $9,318 to $3,068. This was the largest improvement from tuning in the whole assignment.

KNN Regressor (k and weights):
- Smaller k values worked better. With uniform weights, CV R² dropped from 0.7586 at k = 1 to 0.6293 at k = 9. With only 164 training cars, using more neighbours means averaging prices of cars that are not very similar.
- Distance weighting helped at every k above 1, because closer (more similar) cars get a bigger say in the prediction.
- The best setting was k = 3 with distance weights (CV R² 0.7750). Test R² improved from 0.7879 to 0.8723, and RMSE dropped from $4,091 to $3,175.

Which model performs best and why?
After tuning, all three models perform similarly on the test set (R² between 0.8555 and 0.8808). Based on cross-validation, which is more reliable than the small test set, Ridge Regression with alpha = 10 performed best (CV R² 0.8796), followed by SVR with a linear kernel (0.8445) and KNN (0.7750). The linear models (Ridge and linear SVR) worked best because the main features, like engine size, car width, and curb weight, have a mostly straight-line relationship with price. KNN was weaker because the dataset is small, so many cars do not have close neighbours to learn from.

# Conclusion

Classification (Wine):
- All three models performed very well, because the three wine cultivars have clear chemical differences.
- With default settings, all three models reached 97.22% test accuracy.
- After tuning, Logistic Regression with C = 0.1 performed best, with 99.31% cross-validation accuracy and 100% test accuracy.
- Scaling was important for SVC and KNN, because they depend on distances.

Regression (Car Prices):
- With default settings, Linear Regression performed best (test R² 0.8926), KNN was second (0.7879), and SVR failed completely (-0.0999).
- Tuning made a big difference. SVR improved to a test R² of 0.8808 by increasing C, and KNN improved to 0.8723 by using k = 3 with distance weighting.
- Based on cross-validation, Ridge Regression with alpha = 10 was the most reliable model.

What I learned:
- Default hyperparameters are not always suitable. SVR showed this clearly, going from worse than guessing the average to one of the best models after tuning.
- Hyperparameters can cause underfitting when set too strict (very small C or very large alpha) and overfitting when set too loose (very large C or k = 1).
- With small test sets, one prediction can change the results a lot, so cross-validation is a more reliable way to compare models.
- Using a Pipeline made the process safer, because the scaler only learned from the training data during cross-validation.